# ElectroCRM job scoper: QLoRA fine-tune

Fine-tunes a small open model to turn an electrical job description into a structured scope (tasks, labor hours, parts and quantities). Prices are **not** learned; the pricing engine in `ml/src/pricing.py` computes them from the shop's settings, its supplier prices and BLS data.

**Setup:** Settings → Accelerator → **GPU T4 x2** (or P100), Internet **on**. Add the dataset you uploaded from `ml/data/sft/` (train/val/test.jsonl) with "Add Input". Training data is synthetic (generated by a teacher model).

In [ ]:
%%capture
!pip install -U unsloth
# Unpinned for now; once a run works, pin the version printed below (unsloth==X.Y.Z) for repeatable runs.

In [ ]:
import glob, json, os
from importlib.metadata import version
print("unsloth", version("unsloth"), "trl", version("trl"), "transformers", version("transformers"))
# Base model and run settings. Pick any Unsloth 4-bit instruct model that fits the GPU.
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_SEQ = 4096          # the system prompt carries the parts catalog (~2k tokens)
EPOCHS = 2
LR = 2e-4
LORA_R = 16

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found; add the dataset with Add Input"
    return hits[0]

def read_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

train_rows, val_rows, test_rows = (read_jsonl(find(f"{s}.jsonl")) for s in ("train", "val", "test"))
print(len(train_rows), len(val_rows), len(test_rows))

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True, dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_R,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=7,
)

In [ ]:
from datasets import Dataset

def to_text(rows):
    return Dataset.from_list([
        {"text": tokenizer.apply_chat_template(r["messages"], tokenize=False)} for r in rows
    ])

train_ds = to_text(train_rows)
# A tiny dataset can have an empty val split; skip evaluation instead of crashing.
val_ds = to_text(val_rows) if val_rows else None
lengths = [len(tokenizer(t)["input_ids"]) for t in train_ds["text"][:50]]
print("tokens per example (first 50): max", max(lengths), "mean", sum(lengths) // len(lengths))

In [ ]:
# Predictions on the held-out test set, in the format `python -m src.evaluate score` reads.
os.makedirs("/kaggle/working/runs", exist_ok=True)

def predict(path):
    FastLanguageModel.for_inference(model)
    with open(path, "w") as f:
        for i, r in enumerate(test_rows):
            prompt = tokenizer.apply_chat_template(r["messages"][:2], tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
            out = model.generate(**inputs, max_new_tokens=1024, do_sample=False)
            reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            f.write(json.dumps({"id": r["id"], "reply": reply}) + "\n")
            if i % 20 == 0:
                print(i, reply[:200])
    FastLanguageModel.for_training(model)

def valid_json(path):
    ok = 0
    for line in open(path):
        reply = json.loads(line)["reply"]
        try:
            json.loads(reply[reply.find("{"): reply.rfind("}") + 1]); ok += 1
        except Exception:
            pass
    return f"{ok}/{len(test_rows)}"

In [ ]:
# Baseline: the untuned model on the test set. LoRA weights start at zero, so this is the base model.
predict("/kaggle/working/runs/base.jsonl")
print("base valid JSON:", valid_json("/kaggle/working/runs/base.jsonl"))

In [ ]:
import torch
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        logging_steps=10,
        eval_strategy="epoch" if val_ds is not None else "no",
        save_strategy="no",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        seed=7,
        output_dir="outputs",
        report_to="none",
    ),
)
# Only learn from the assistant's JSON, not the long system prompt.
# These markers are for Qwen's ChatML template; change them if you change BASE_MODEL.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
stats = trainer.train()
print(stats)

In [ ]:
predict("/kaggle/working/runs/finetuned.jsonl")
print("base      valid JSON:", valid_json("/kaggle/working/runs/base.jsonl"))
print("finetuned valid JSON:", valid_json("/kaggle/working/runs/finetuned.jsonl"))
print("Full report: download runs/ and run `python -m src.evaluate score runs/<file>.jsonl` locally.")

In [ ]:
# Save the LoRA adapter and a q4_k_m GGUF for Ollama (runs on a 6 GB GPU or CPU).
import shutil
model.save_pretrained("/kaggle/working/scoper-lora")
tokenizer.save_pretrained("/kaggle/working/scoper-lora")
model.save_pretrained_gguf("/kaggle/working/scoper-gguf", tokenizer, quantization_method="q4_k_m")
# The GGUF export leaves a ~6 GB merged 16-bit copy and caches behind; keep only what we download.
for leftover in ("scoper-gguf", "unsloth_compiled_cache", "huggingface_tokenizers_cache", "outputs"):
    shutil.rmtree(f"/kaggle/working/{leftover}", ignore_errors=True)
!ls -laR /kaggle/working